In [23]:
!git clone https://github.com/Priyansh-81/Architectural-Immunology.git
%cd Architectural-Immunology
!pip install -U transformers accelerate bitsandbytes sentencepiece datasets

Cloning into 'Architectural-Immunology'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 84 (delta 41), reused 51 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 53.11 KiB | 4.83 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/Architectural-Immunology/Architectural-Immunology


In [24]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [25]:
import os
from getpass import getpass

token = getpass("HF token: ")

if token:
    os.environ["HF_TOKEN"] = token

HF token: ··········


In [26]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [27]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [28]:
small_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

small_tokenizer = AutoTokenizer.from_pretrained(small_model_name)

small_model = AutoModelForCausalLM.from_pretrained(
    small_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Small model loaded")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Small model loaded


In [29]:
reasoning_model_name = "Qwen/Qwen2.5-3B-Instruct"

reasoning_tokenizer = AutoTokenizer.from_pretrained(reasoning_model_name)

reasoning_model = AutoModelForCausalLM.from_pretrained(
    reasoning_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Reasoning model loaded")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Reasoning model loaded


In [30]:
print(
    "GPU memory allocated:",
    torch.cuda.memory_allocated()/1e9,
    "GB"
)

GPU memory allocated: 7.17065984 GB


In [31]:
prompt = "Question: What is 2+2?\nAnswer:"

inputs = reasoning_tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = reasoning_model.generate(
    **inputs,
    max_new_tokens=30
)

print(
    reasoning_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Question: What is 2+2?
Answer: 4

Question: What is the capital of France?
Answer: Paris

Question: How many continents are there in the world?
Answer: There


In [32]:
!pip install datasets pandas

In [33]:
from huggingface_hub import login
login()

In [34]:
from datasets import load_dataset

In [35]:
dataset = load_dataset("gaia-benchmark/GAIA", "2023_level1")

In [36]:
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['task_id', 'Question', 'Level', 'Final answer', 'file_name', 'file_path', 'Annotator Metadata'],
        num_rows: 93
    })
    validation: Dataset({
        features: ['task_id', 'Question', 'Level', 'Final answer', 'file_name', 'file_path', 'Annotator Metadata'],
        num_rows: 53
    })
})


In [37]:
from src.models import load_models
small_model, reasoning_model = load_models()

IndentationError: unexpected indent (models.py, line 1)